# IDH por estado — melt e Matplotlib

A Tabela 4 é a planilha anexa da aula. Os números abaixo são os dessa tabela, não uma consulta externa.

A primeira linha do CSV é o título. O cabeçalho começa na linha seguinte, com separador `;` e vírgula decimal.

Execute a partir da pasta da atividade ou da raiz de `analise_dados`.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

def raiz_do_repositorio():
    aqui = Path.cwd().resolve()
    for pasta in [aqui, *aqui.parents]:
        if (pasta / "dados").is_dir() and (pasta / "etapa_3").is_dir():
            return pasta
    raise FileNotFoundError("Execute a partir do repositorio analise_dados.")

RAIZ = raiz_do_repositorio()
DADOS = RAIZ / "dados"
SAIDAS = RAIZ / "etapa_3" / "05_idh_melt" / "saidas"
SAIDAS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DADOS / "Tabela4.csv",
    sep=";",
    decimal=",",
    encoding="cp1252",
    skiprows=1,
)
df = df.dropna(axis=1, how="all")
df = df.dropna(how="all")
df.columns = [str(coluna).strip() for coluna in df.columns]
print(df.columns.tolist())
print("linhas:", len(df))
df.head()


['Sigla', 'Código', 'Estado', '1991', '2000', '2010', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
linhas: 27


,Sigla,Código,Estado,1991,2000,2010,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,AC,12,Acre,0.402,0.517,0.663,0.705,0.709,0.719,0.719,0.723,0.711,0.731,0.738,0.736,0.702,0.732,0.740,0.754
1,AL,27,Alagoas,0.370,0.471,0.631,0.657,0.653,0.670,0.669,0.680,0.682,0.694,0.693,0.694,0.691,0.722,0.739,0.746
2,AM,13,Amazonas,0.430,0.515,0.674,0.706,0.717,0.720,0.724,0.723,0.743,0.732,0.739,0.725,0.700,0.757,0.764,0.767
3,AP,16,Amapá,0.472,0.577,0.708,0.709,0.735,0.726,0.728,0.732,0.731,0.737,0.732,0.707,0.681,0.743,0.767,0.759
4,BA,29,Bahia,0.386,0.512,0.660,0.684,0.686,0.700,0.708,0.708,0.715,0.714,0.723,0.726,0.698,0.734,0.740,0.759


In [2]:
anos = [coluna for coluna in df.columns if str(coluna).isdigit()]
print("anos:", anos)

id_vars = [coluna for coluna in df.columns if coluna not in anos]
ranking_2024 = df.sort_values("2024", ascending=False)[["Sigla", "Estado", "2024"]]
print("Estados por IDH de 2024, do maior para o menor:")
ranking_2024


anos: ['1991', '2000', '2010', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Estados por IDH de 2024, do maior para o menor:


,Sigla,Estado,2024
6,DF,Distrito Federal,0.866
25,SP,São Paulo,0.838
23,SC,Santa Catarina,0.833
17,PR,Paraná,0.822
18,RJ,Rio de Janeiro,0.819
22,RS,Rio Grande do Sul,0.818
8,GO,Goiás,0.815
12,MT,Mato Grosso,0.812
10,MG,Minas Gerais,0.809
7,ES,Espírito Santo,0.804


In [3]:
df["melhora_1991_2024"] = df["2024"] - df["1991"]
maior = df.loc[df["melhora_1991_2024"].idxmax()]
print(
    f"Maior melhora entre 1991 e 2024: {maior['Estado']} "
    f"({maior['Sigla']}), {maior['melhora_1991_2024']:.3f} ponto de IDH."
)

piora_periodo = df.loc[df["melhora_1991_2024"] < 0, ["Sigla", "Estado", "1991", "2024"]]
print("Estados com IDH de 2024 menor que o de 1991:", len(piora_periodo))
piora_periodo


Maior melhora entre 1991 e 2024: Tocantins (TO), 0.428 ponto de IDH.
Estados com IDH de 2024 menor que o de 1991: 0


,Sigla,Estado,1991,2024


In [4]:
anos_ord = sorted(anos, key=int)
quedas = []
for _, linha in df.iterrows():
    for anterior, atual in zip(anos_ord, anos_ord[1:]):
        if pd.notna(linha[atual]) and pd.notna(linha[anterior]) and linha[atual] < linha[anterior]:
            quedas.append(
                {
                    "Sigla": linha["Sigla"],
                    "de": int(anterior),
                    "para": int(atual),
                    "queda": linha[anterior] - linha[atual],
                }
            )

quedas = pd.DataFrame(quedas)
print("Quedas de um ano da tabela para o seguinte:", len(quedas))
print("Estados com alguma queda intermediária:", quedas["Sigla"].nunique())
print("Nenhum valor ausente foi trocado por zero.")
quedas.sort_values("queda", ascending=False).head(10)


Quedas de um ano da tabela para o seguinte: 84
Estados com alguma queda intermediária: 27
Nenhum valor ausente foi trocado por zero.


,Sigla,de,para,queda
62,RO,2020,2021,0.044
68,RR,2020,2021,0.041
2,AC,2020,2021,0.034
83,TO,2020,2021,0.029
18,BA,2020,2021,0.028
54,PR,2020,2021,0.026
15,AP,2020,2021,0.026
10,AM,2020,2021,0.025
14,AP,2019,2020,0.025
22,DF,2019,2020,0.023


Entre 1991 e 2024 a comparação é a coluna de melhora. As quedas intermediárias respondem outra pergunta: houve ano em que o IDH de um estado ficou abaixo do ano anterior da tabela. As duas contas ficam separadas.


In [5]:
df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)
df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")

print("ufs:", df_longo["Sigla"].nunique())
print("anos:", sorted(df_longo["Ano"].unique()))
print("duplicidade UF/ano:", int(df_longo.duplicated(["Sigla", "Ano"]).sum()))
print("IDH ausente:", int(df_longo["IDH"].isna().sum()))
df_longo.head()


ufs:

 27
anos: [np.int64(1991), np.int64(2000), np.int64(2010), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
duplicidade UF/ano: 0
IDH ausente: 0


,Sigla,Código,Estado,Ano,IDH
0,AC,12,Acre,1991,0.402
1,AL,27,Alagoas,1991,0.370
2,AM,13,Amazonas,1991,0.430
3,AP,16,Amapá,1991,0.472
4,BA,29,Bahia,1991,0.386


In [6]:
minas = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(minas["Ano"], minas["IDH"], marker="o")
ax.set_title("Evolução do IDH — Minas Gerais")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
plt.tight_layout()
fig.savefig(SAIDAS / "idh_minas_gerais.png", dpi=120, bbox_inches="tight")
plt.show()


C:\Users\alzat\AppData\Local\Temp\ipykernel_4048\2446620654.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print("IDH mínimo:", df_longo["IDH"].min(), "máximo:", df_longo["IDH"].max())

fig, ax = plt.subplots(figsize=(12, 7))
for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(grupo["Ano"], grupo["IDH"], marker="o", markersize=3, linewidth=1.5, label=sigla)

ax.set_title("Evolução do IDH por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDH")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
fig.savefig(SAIDAS / "idh_todos_estados.png", dpi=120, bbox_inches="tight")
plt.show()

IDH mínimo: 0.357 máximo: 0.87


C:\Users\alzat\AppData\Local\Temp\ipykernel_4048\3095922541.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
